In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, math, random
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/Agatha_Christie/61262-0.txt")
assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"
print("Found:", DATA_PATH)

Mounted at /content/drive
Found: /content/drive/MyDrive/Agatha_Christie/61262-0.txt


# Quick Testing

In [2]:
def strip_gutenberg_boilerplate(text: str) -> str:
    # Common Gutenberg markers (not guaranteed in every file)
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG EBOOK",
        "***START OF THIS PROJECT GUTENBERG EBOOK",
        "*** START OF THE PROJECT GUTENBERG EBOOK",
    ]
    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG EBOOK",
        "***END OF THIS PROJECT GUTENBERG EBOOK",
        "*** END OF THE PROJECT GUTENBERG EBOOK",
    ]

    lower = text
    start_idx = None
    for m in start_markers:
        i = lower.find(m)
        if i != -1:
            start_idx = i
            break

    end_idx = None
    for m in end_markers:
        i = lower.find(m)
        if i != -1:
            end_idx = i
            break

    if start_idx is not None:
        # start after the marker line
        after = lower.find("\n", start_idx)
        start_idx = after if after != -1 else start_idx
    else:
        start_idx = 0

    if end_idx is None:
        end_idx = len(text)

    cleaned = text[start_idx:end_idx]
    cleaned = cleaned.strip()
    return cleaned

raw_text = DATA_PATH.read_text(encoding="utf-8", errors="replace")
text = strip_gutenberg_boilerplate(raw_text)

print("Chars (raw):", len(raw_text))
print("Chars (clean):", len(text))
print("\nPreview:\n", text[:800])

Chars (raw): 296991
Chars (clean): 296886

Preview:
 POIROT INVESTIGATES




  BY THE SAME AUTHOR

  THE MYSTERIOUS AFFAIR AT STYLES

  THE SECRET ADVERSARY

  THE MURDER ON THE LINKS

  THE BODLEY HEAD




  POIROT INVESTIGATES

  BY AGATHA CHRISTIE




  LONDON

  JOHN LANE THE BODLEY HEAD LIMITED




  First published in Great Britain by
  John Lane Company, The Bodley Head Limited, 1924

  Copyright © 1924 Agatha Christie Limited




  CONTENTS


  I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The Adventure of the Egyptian Tomb

  VII Jewel Robbery at the _Grand Metropolitan_

  VIII The Kidnapped Prime Minister

  IX The Disappearance of Mr. Davenheim

  X The Adventure of the Italian Nobleman



In [3]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text: str):
    # Keeps punctuation as its own token; splits on whitespace implicitly.
    return TOKEN_RE.findall(text)

tokens = tokenize(text)
print("Num tokens:", len(tokens))
print("First 80 tokens:", tokens[:80])

Num tokens: 68373
First 80 tokens: ['POIROT', 'INVESTIGATES', 'BY', 'THE', 'SAME', 'AUTHOR', 'THE', 'MYSTERIOUS', 'AFFAIR', 'AT', 'STYLES', 'THE', 'SECRET', 'ADVERSARY', 'THE', 'MURDER', 'ON', 'THE', 'LINKS', 'THE', 'BODLEY', 'HEAD', 'POIROT', 'INVESTIGATES', 'BY', 'AGATHA', 'CHRISTIE', 'LONDON', 'JOHN', 'LANE', 'THE', 'BODLEY', 'HEAD', 'LIMITED', 'First', 'published', 'in', 'Great', 'Britain', 'by', 'John', 'Lane', 'Company', ',', 'The', 'Bodley', 'Head', 'Limited', ',', '1924', 'Copyright', '©', '1924', 'Agatha', 'Christie', 'Limited', 'CONTENTS', 'I', 'The', 'Adventure', 'of', '“', 'The', 'Western', 'Star', '”', 'II', 'The', 'Tragedy', 'at', 'Marsdon', 'Manor', 'III', 'The', 'Adventure', 'of', 'the', 'Cheap', 'Flat', 'IV']


In [4]:
val_frac = 0.10
n = len(tokens)
n_val = int(n * val_frac)

train_tokens = tokens[:-n_val]
val_tokens = tokens[-n_val:]

print("Train tokens:", len(train_tokens))
print("Val tokens:", len(val_tokens))
print("Val starts with:", val_tokens[:40])

Train tokens: 61536
Val tokens: 6837
Val starts with: ['agitation', '.', 'This', 'was', 'Graves', ',', 'valet', '-', 'butler', 'to', 'the', 'late', 'Count', 'Foscatini', '.', 'The', 'story', 'he', 'had', 'to', 'tell', 'was', 'a', 'sensational', 'one', '.', 'On', 'the', 'previous', 'morning', ',', 'two', 'gentlemen', 'had', 'called', 'to', 'see', 'his', 'master', '.']


In [5]:
from collections import Counter

PAD = "<PAD>"
UNK = "<UNK>"

min_freq = 2

counter = Counter(train_tokens)
vocab_tokens = [tok for tok, c in counter.items() if c >= min_freq]
vocab_tokens = sorted(vocab_tokens, key=lambda t: (-counter[t], t))

itos = [PAD, UNK] + vocab_tokens
stoi = {t:i for i,t in enumerate(itos)}

def encode(tok_list):
    return [stoi.get(t, stoi[UNK]) for t in tok_list]

def decode(id_list):
    return [itos[i] for i in id_list]

train_ids = encode(train_tokens)
val_ids = encode(val_tokens)

print("Vocab size:", len(itos))
unk_rate = sum(1 for t in train_tokens if t not in stoi) / len(train_tokens)
print("Train UNK rate (should be ~0 because vocab built on train):", unk_rate)

val_unk_rate = sum(1 for t in val_tokens if t not in stoi) / len(val_tokens)
print("Val UNK rate:", val_unk_rate)

Vocab size: 3038
Train UNK rate (should be ~0 because vocab built on train): 0.050149505980239206
Val UNK rate: 0.11028228755302033


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class LMDataset(Dataset):
    def __init__(self, ids, block_size: int):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        # number of possible windows
        return max(0, len(self.ids) - self.block_size - 1)

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.block_size]
        y = self.ids[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 128  # good starting point
train_ds = LMDataset(train_ids, block_size)
val_ds = LMDataset(val_ids, block_size)

print("Train windows:", len(train_ds))
print("Val windows:", len(val_ds))

# quick sanity sample
x0, y0 = train_ds[0]
print("x0 shape:", x0.shape, "y0 shape:", y0.shape)
print("x0 tokens:", decode(x0[:30].tolist()))
print("y0 tokens:", decode(y0[:30].tolist()))

Train windows: 61407
Val windows: 6708
x0 shape: torch.Size([128]) y0 shape: torch.Size([128])
x0 tokens: ['POIROT', 'INVESTIGATES', 'BY', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', 'THE', 'BODLEY', 'HEAD', 'POIROT', 'INVESTIGATES', 'BY', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
y0 tokens: ['INVESTIGATES', 'BY', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', '<UNK>', 'THE', '<UNK>', 'THE', 'BODLEY', 'HEAD', 'POIROT', 'INVESTIGATES', 'BY', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', 'THE']


In [7]:
batch_size = 64  # adjust later based on speed/VRAM

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

xb, yb = next(iter(train_loader))
print("Batch x:", xb.shape, "Batch y:", yb.shape)
print("Example (decoded):")
print("X:", " ".join(decode(xb[0][:40].tolist())))
print("Y:", " ".join(decode(yb[0][:40].tolist())))

Batch x: torch.Size([64, 128]) Batch y: torch.Size([64, 128])
Example (decoded):
X: face , and I gave a start of surprise . “ He ’ s not a Jap , ” I ejaculated in a whisper to Poirot . “ <UNK> was always your strong point , Hastings ! Nothing <UNK> you
Y: , and I gave a start of surprise . “ He ’ s not a Jap , ” I ejaculated in a whisper to Poirot . “ <UNK> was always your strong point , Hastings ! Nothing <UNK> you .


In [8]:
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class GRULM(nn.Module):
    def __init__(self, vocab_size, d_model=256, hidden=512, n_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
        )
        self.drop = nn.Dropout(dropout)
        self.lm_head = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        # x: (B, T)
        e = self.embed(x)               # (B, T, d_model)
        h, _ = self.gru(e)              # (B, T, hidden)
        h = self.drop(h)
        logits = self.lm_head(h)        # (B, T, V)
        return logits

model = GRULM(vocab_size=len(itos)).to(device)
num_params = sum(p.numel() for p in model.parameters())
print("params:", f"{num_params/1e6:.2f}M")

device: cuda
params: 5.09M


In [9]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler(enabled=(device == "cuda"))

def run_epoch(model, loader, optimizer=None, clip_grad=1.0):
    train = optimizer is not None
    model.train(train)

    total_loss = 0.0
    total_tokens = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)  # (B, T)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device == "cuda")):
            logits = model(xb)  # (B, T, V)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                yb.reshape(-1),
            )

        if train:
            scaler.scale(loss).backward()
            if clip_grad is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            scaler.step(optimizer)
            scaler.update()

        B, T = yb.shape
        total_loss += loss.item() * (B * T)
        total_tokens += (B * T)

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, ppl

/tmp/ipython-input-3905811468.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))


In [10]:
import math
import torch.optim as optim

lr = 3e-4
optimizer = optim.AdamW(model.parameters(), lr=lr)

epochs = 3  # baseline: keep short
for ep in range(1, epochs + 1):
    train_loss, train_ppl = run_epoch(model, train_loader, optimizer=optimizer, clip_grad=1.0)
    val_loss, val_ppl = run_epoch(model, val_loader, optimizer=None)

    print(f"epoch {ep:02d} | train loss {train_loss:.4f} ppl {train_ppl:.2f} | val loss {val_loss:.4f} ppl {val_ppl:.2f}")

/tmp/ipython-input-3905811468.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):


epoch 01 | train loss 4.2163 ppl 67.78 | val loss 4.4108 ppl 82.34
epoch 02 | train loss 2.2158 ppl 9.17 | val loss 5.9741 ppl 393.11
epoch 03 | train loss 0.9504 ppl 2.59 | val loss 7.4255 ppl 1678.30


In [11]:
def sample_next_token(logits, temperature=1.0, top_k=0, top_p=1.0):
    # logits: (V,)
    if temperature <= 0:
        return torch.argmax(logits).item()

    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    # top-k
    if top_k and top_k > 0:
        v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
        mask = torch.zeros_like(probs)
        mask[idx] = v
        probs = mask / mask.sum()

    # top-p (nucleus)
    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cum > top_p
        cutoff[..., 0] = False
        sorted_probs[cutoff] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        next_id = sorted_idx[torch.multinomial(sorted_probs, 1)].item()
        return next_id

    return torch.multinomial(probs, 1).item()

In [13]:
def detokenize(tok_list):
    # Simple English-ish detokenisation
    out = []
    no_space_before = {".", ",", "!", "?", ";", ":", ")", "”", "’", "'"}
    no_space_after = {"(", "“"}
    for t in tok_list:
        if not out:
            out.append(t)
            continue
        if t in no_space_before:
            out[-1] = out[-1] + t
        elif out[-1] in no_space_after:
            out[-1] = out[-1] + t
        else:
            out.append(" " + t)
    return "".join(out)

In [14]:
@torch.no_grad()
def generate(model, prompt_tokens, max_new_tokens=200, temperature=0.9, top_k=50, top_p=0.95):
    model.eval()
    ids = encode(prompt_tokens)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, T)

    for _ in range(max_new_tokens):
        # Feed the last block_size tokens
        x_cond = x[:, -block_size:]
        logits = model(x_cond)               # (1, T, V)
        next_logits = logits[0, -1]          # (V,)
        next_id = sample_next_token(next_logits, temperature, top_k, top_p)

        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)

    return decode(x[0].tolist())

seed = input("Enter a few words (prompt): ").strip()
prompt_tokens = tokenize(seed) if seed else ["Poirot"]
gen_tokens = generate(model, prompt_tokens, max_new_tokens=220, temperature=0.9, top_k=50, top_p=0.95)

print("\n--- Generated ---\n")
print(detokenize(gen_tokens))

Enter a few words (prompt): The person standing on the window jumped

--- Generated ---

The person standing on the window jumped at once.” “ What is his little rook rifle now?” “ Thank you, Captain Black. Perhaps you went to me.” “ You see nothing?” “ I’ ve been getting serious.” “ It seems to me,” I said, “ that that <UNK> everything. What a horrible crime!” He waved his sponge. I smiled as I lit another cigarette. “ <UNK> interesting on?” I inquired, after a minute or two. “ Unfortunately, my friend, there are some things that cannot be sent by telegram.” <UNK> the <UNK> authorities, the door. As he left her hand. “ I’ m afraid I don’ t. Just there with a faint <UNK> hint, my friend. I was smiling in Count Foscatini.” She was about five - door. The door was not a <UNK> like to light up to turn into the <UNK> and <UNK>. I felt that at last Poirot had <UNK> his <UNK>, and she was <UNK> cheap of his <UNK>’ s arm. It was: “ The Conference is to say that I have never heard of a joke.


In [15]:
import re

def trim_to_story(text: str) -> str:
    candidates = [
        r"\nI\s+The\s+Adventure\s+of",    # common contents heading
        r"\nI\.\s+The\s+Adventure\s+of",
        r"\nCHAPTER\s+I",
        r"\nChapter\s+I",
        r"\nTHE\s+ADVENTURE\s+OF",
        r"\nThe\s+Adventure\s+of",
    ]
    best = None
    for pat in candidates:
        m = re.search(pat, text)
        if m:
            best = m.start()
            break
    if best is None:
        return text.strip()
    return text[best:].strip()

text2 = trim_to_story(text)
print("Chars before:", len(text))
print("Chars after :", len(text2))
print("Preview:\n", text2[:600])

Chars before: 296886
Chars after : 296886
Preview:
 POIROT INVESTIGATES




  BY THE SAME AUTHOR

  THE MYSTERIOUS AFFAIR AT STYLES

  THE SECRET ADVERSARY

  THE MURDER ON THE LINKS

  THE BODLEY HEAD




  POIROT INVESTIGATES

  BY AGATHA CHRISTIE




  LONDON

  JOHN LANE THE BODLEY HEAD LIMITED




  First published in Great Britain by
  John Lane Company, The Bodley Head Limited, 1924

  Copyright © 1924 Agatha Christie Limited




  CONTENTS


  I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The A


In [16]:
tokens2 = tokenize(text2)
print("Num tokens (trimmed):", len(tokens2))
print("First 80 tokens:", tokens2[:80])

Num tokens (trimmed): 68373
First 80 tokens: ['POIROT', 'INVESTIGATES', 'BY', 'THE', 'SAME', 'AUTHOR', 'THE', 'MYSTERIOUS', 'AFFAIR', 'AT', 'STYLES', 'THE', 'SECRET', 'ADVERSARY', 'THE', 'MURDER', 'ON', 'THE', 'LINKS', 'THE', 'BODLEY', 'HEAD', 'POIROT', 'INVESTIGATES', 'BY', 'AGATHA', 'CHRISTIE', 'LONDON', 'JOHN', 'LANE', 'THE', 'BODLEY', 'HEAD', 'LIMITED', 'First', 'published', 'in', 'Great', 'Britain', 'by', 'John', 'Lane', 'Company', ',', 'The', 'Bodley', 'Head', 'Limited', ',', '1924', 'Copyright', '©', '1924', 'Agatha', 'Christie', 'Limited', 'CONTENTS', 'I', 'The', 'Adventure', 'of', '“', 'The', 'Western', 'Star', '”', 'II', 'The', 'Tragedy', 'at', 'Marsdon', 'Manor', 'III', 'The', 'Adventure', 'of', 'the', 'Cheap', 'Flat', 'IV']


In [17]:
val_frac = 0.10
n = len(tokens2)
n_val = int(n * val_frac)

train_tokens2 = tokens2[:-n_val]
val_tokens2 = tokens2[-n_val:]

print("Train tokens:", len(train_tokens2))
print("Val tokens:", len(val_tokens2))
print("Val starts with:", val_tokens2[:40])

Train tokens: 61536
Val tokens: 6837
Val starts with: ['agitation', '.', 'This', 'was', 'Graves', ',', 'valet', '-', 'butler', 'to', 'the', 'late', 'Count', 'Foscatini', '.', 'The', 'story', 'he', 'had', 'to', 'tell', 'was', 'a', 'sensational', 'one', '.', 'On', 'the', 'previous', 'morning', ',', 'two', 'gentlemen', 'had', 'called', 'to', 'see', 'his', 'master', '.']


In [18]:
from collections import Counter

PAD = "<PAD>"
UNK = "<UNK>"

counter2 = Counter(train_tokens2)
# keep ALL tokens seen in train
vocab_tokens2 = sorted(counter2.keys(), key=lambda t: (-counter2[t], t))

itos2 = [PAD, UNK] + vocab_tokens2
stoi2 = {t:i for i,t in enumerate(itos2)}

def encode2(tok_list):
    return [stoi2.get(t, stoi2[UNK]) for t in tok_list]

def decode2(id_list):
    return [itos2[i] for i in id_list]

train_ids2 = encode2(train_tokens2)
val_ids2 = encode2(val_tokens2)

print("Vocab size:", len(itos2))
val_unk_rate = sum(1 for t in val_tokens2 if t not in stoi2) / len(val_tokens2)
print("Val UNK rate:", val_unk_rate)


Vocab size: 6124
Val UNK rate: 0.07430159426649115


In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

class LMDataset(Dataset):
    def __init__(self, ids, block_size: int):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size - 1)

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.block_size]
        y = self.ids[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 128
train_ds2 = LMDataset(train_ids2, block_size)
val_ds2 = LMDataset(val_ids2, block_size)

batch_size = 64
train_loader2 = DataLoader(train_ds2, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader2 = DataLoader(val_ds2, batch_size=batch_size, shuffle=False, drop_last=True)

print("Train windows:", len(train_ds2))
print("Val windows:", len(val_ds2))

xb, yb = next(iter(train_loader2))
print("Batch x:", xb.shape, "Batch y:", yb.shape)

Train windows: 61407
Val windows: 6708
Batch x: torch.Size([64, 128]) Batch y: torch.Size([64, 128])


In [21]:
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, nh, T, T)
        mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        att = att.masked_fill(~mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v  # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y

device: cuda


In [22]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout, mlp_ratio=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [23]:
class GPTSmall(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=256, n_layers=4, n_heads=4, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # weight tying (helps a bit with small data)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.block_size
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # (1, T)

        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, V)
        return logits

model = GPTSmall(vocab_size=len(itos2), block_size=block_size, d_model=256, n_layers=4, n_heads=4, dropout=0.25).to(device)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

params: 4.76M


In [24]:
import torch.optim as optim
from torch.amp import autocast, GradScaler  # avoids the deprecation warning

scaler = GradScaler(enabled=(device == "cuda"))

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def eval_loss(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device_type="cuda", enabled=(device=="cuda")):
                logits = model(xb)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))
            B, T = yb.shape
            total_loss += loss.item() * (B*T)
            total_tokens += (B*T)
    avg = total_loss / total_tokens
    ppl = math.exp(avg) if avg < 20 else float("inf")
    return avg, ppl

max_steps = 1500         # usually well within time on A100 for this size
eval_every = 200
clip_grad = 1.0

model.train()
best_val = float("inf")

step = 0
it = iter(train_loader2)

while step < max_steps:
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(train_loader2)
        xb, yb = next(it)

    xb, yb = xb.to(device), yb.to(device)

    optimizer.zero_grad(set_to_none=True)
    with autocast(device_type="cuda", enabled=(device=="cuda")):
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % eval_every == 0:
        train_loss = loss.item()
        val_loss, val_ppl = eval_loss(model, val_loader2)
        print(f"step {step:04d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} ppl {val_ppl:.2f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

step 0200 | train loss 11.6475 | val loss 9.1101 ppl 9046.00
step 0400 | train loss 6.1116 | val loss 6.4564 ppl 636.79
step 0600 | train loss 5.5141 | val loss 6.1186 ppl 454.25
step 0800 | train loss 5.3038 | val loss 5.9997 ppl 403.29
step 1000 | train loss 5.0890 | val loss 6.0153 ppl 409.65
step 1200 | train loss 4.8943 | val loss 6.0227 ppl 412.70
step 1400 | train loss 4.7428 | val loss 6.0442 ppl 421.65


In [25]:
# restore best checkpoint (if we saved one)
if "best_state" in globals():
    model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
    print("Loaded best checkpoint with val loss:", best_val)

Loaded best checkpoint with val loss: 5.999653133062216


In [26]:
def sample_next_token(logits, temperature=1.0, top_k=0, top_p=1.0):
    if temperature <= 0:
        return torch.argmax(logits).item()
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    if top_k and top_k > 0:
        v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
        mask = torch.zeros_like(probs)
        mask[idx] = v
        probs = mask / mask.sum()

    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cum > top_p
        cutoff[..., 0] = False
        sorted_probs[cutoff] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        return sorted_idx[torch.multinomial(sorted_probs, 1)].item()

    return torch.multinomial(probs, 1).item()

In [27]:
@torch.no_grad()
def generate_tokens(prompt, max_new_tokens=220, temperature=0.9, top_k=50, top_p=0.95):
    model.eval()
    prompt_toks = tokenize(prompt)
    ids = [stoi2.get(t, stoi2[UNK]) for t in prompt_toks]
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits = model(x_cond)
        next_id = sample_next_token(logits[0, -1], temperature, top_k, top_p)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)

    toks = [itos2[i] for i in x[0].tolist()]
    return toks

seed = input("Enter a few words (prompt): ").strip()
if not seed:
    seed = "Poirot"

out = generate_tokens(seed, max_new_tokens=260, temperature=0.9, top_k=50, top_p=0.95)
print("\n--- Generated ---\n")
print(detokenize(out))

Enter a few words (prompt): the man jumped from the window

--- Generated ---

the man jumped from the window. Now. But and all one of the. at once.” “ I don’ s.” “ And a man who’ t to London and that I have been Yardly,” “ My have a moment I the Prime Minister,” “ What the.” “ I cried. But there, but I will you know, but him from the head is no not in a You do you to do me in the good good. But he was come to find a man, the window?” “ But there are’ s,” “ I have the window. “ I’ s one.” said’ s all it was a the ‘ are at him. “ Well of the moment a man like for the window of it to me in a few minutes, if you, is one it was of the by the room.” “ And a ‘ are of.” “ He has something you will and I was a little man was of the two. I’ s voice.” I understand you, is you, and then, or two been’ s the voice. I see. Maltravers, and you can for the man that of the moment the little man, the Prime Minister in my friend, I am’ t know?” “ I am Farquhar of the Prime Minister and


In [35]:
import re

def find_narrative_start(text: str):
    # 1) start searching after CONTENTS
    m = re.search(r"\bCONTENTS\b", text)
    start = m.end() if m else 0
    tail = text[start:]

    # 2) locate the first "I The Adventure of" after CONTENTS (contents list area)
    mm = re.search(r"\nI\s+The\s+Adventure\s+of", tail)
    if not mm:
        # fallback: just scan from start
        tail = text
        mm = re.search(r"\nI\s+The\s+Adventure\s+of", tail)

    anchor = mm.start() if mm else 0
    tail2 = tail[anchor:]

    # 3) split into lines and find first "narrative-like" line
    lines = tail2.splitlines()
    # heuristic: narrative lines have >= 30 chars and lots of lowercase letters
    def narrative_score(line):
        lc = sum(ch.islower() for ch in line)
        return lc

    candidates = []
    for i, line in enumerate(lines):
        if len(line) < 30:
            continue
        if narrative_score(line) >= 20 and (" " in line):
            candidates.append(i)
            if len(candidates) >= 5:
                break

    # show a few candidate contexts
    print("Anchor found:", mm is not None)
    print("First 30 lines after anchor:\n")
    for j in range(min(30, len(lines))):
        print(f"{j:02d}: {lines[j][:120]}")

    print("\nCandidate narrative line indices:", candidates)
    for idx in candidates[:3]:
        lo = max(0, idx-3)
        hi = min(len(lines), idx+6)
        print("\n--- Candidate start context ---")
        for k in range(lo, hi):
            print(f"{k:04d}: {lines[k]}")
    return lines, candidates

lines, candidates = find_narrative_start(text)

Anchor found: False
First 30 lines after anchor:

00: POIROT INVESTIGATES
01: 
02: 
03: 
04: 
05:   BY THE SAME AUTHOR
06: 
07:   THE MYSTERIOUS AFFAIR AT STYLES
08: 
09:   THE SECRET ADVERSARY
10: 
11:   THE MURDER ON THE LINKS
12: 
13:   THE BODLEY HEAD
14: 
15: 
16: 
17: 
18:   POIROT INVESTIGATES
19: 
20:   BY AGATHA CHRISTIE
21: 
22: 
23: 
24: 
25:   LONDON
26: 
27:   JOHN LANE THE BODLEY HEAD LIMITED
28: 
29: 

Candidate narrative line indices: [32, 33, 35, 43, 45]

--- Candidate start context ---
0029: 
0030: 
0031: 
0032:   First published in Great Britain by
0033:   John Lane Company, The Bodley Head Limited, 1924
0034: 
0035:   Copyright © 1924 Agatha Christie Limited
0036: 
0037: 

--- Candidate start context ---
0030: 
0031: 
0032:   First published in Great Britain by
0033:   John Lane Company, The Bodley Head Limited, 1924
0034: 
0035:   Copyright © 1924 Agatha Christie Limited
0036: 
0037: 
0038: 

--- Candidate start context ---
0032:   First published in Great Britain 

In [36]:
tokens_story = tokenize(text_story)

val_frac = 0.10
n = len(tokens_story)
n_val = int(n * val_frac)

train_tokens = tokens_story[:-n_val]
val_tokens   = tokens_story[-n_val:]

from collections import Counter
PAD, UNK = "<PAD>", "<UNK>"
counter = Counter(train_tokens)

# Keep all train tokens (min_freq=1) to reduce <UNK> damage on tiny data
vocab = sorted(counter.keys(), key=lambda t: (-counter[t], t))
itos = [PAD, UNK] + vocab
stoi = {t:i for i,t in enumerate(itos)}

def encode(tok_list): return [stoi.get(t, stoi[UNK]) for t in tok_list]
def decode(id_list): return [itos[i] for i in id_list]

train_ids = encode(train_tokens)
val_ids   = encode(val_tokens)

print("Tokens:", len(tokens_story))
print("Vocab size:", len(itos))
print("Val UNK rate:", sum(1 for t in val_tokens if t not in stoi)/len(val_tokens))

block_size = 128
train_ds = LMDataset(train_ids, block_size)
val_ds   = LMDataset(val_ids, block_size)

from torch.utils.data import DataLoader
batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

print("Train windows:", len(train_ds), "Val windows:", len(val_ds))

Tokens: 68316
Vocab size: 6093
Val UNK rate: 0.07480603132777046
Train windows: 61356 Val windows: 6702


In [37]:
!pip -q install transformers accelerate

import torch
from transformers import GPT2Config, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

config = GPT2Config(
    vocab_size=len(itos),
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=256,
    n_layer=4,
    n_head=4,
    resid_pdrop=0.25,
    embd_pdrop=0.25,
    attn_pdrop=0.25,
    bos_token_id=stoi.get("<BOS>", None),
    eos_token_id=stoi.get("<EOS>", None),
)

model = GPT2LMHeadModel(config).to(device)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

device: cuda
params: 4.75M


In [38]:
import math
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler

scaler = GradScaler(enabled=(device=="cuda"))
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def eval_loss(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device_type="cuda", enabled=(device=="cuda")):
                out = model(input_ids=xb, labels=yb)
                loss = out.loss
            B, T = yb.shape
            total_loss += loss.item() * (B*T)
            total_tokens += (B*T)
    avg = total_loss / total_tokens
    ppl = math.exp(avg) if avg < 20 else float("inf")
    return avg, ppl

max_steps = 1500
eval_every = 200
clip_grad = 1.0

model.train()
best_val = float("inf")
best_state = None

it = iter(train_loader)
for step in range(1, max_steps+1):
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(train_loader)
        xb, yb = next(it)

    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", enabled=(device=="cuda")):
        out = model(input_ids=xb, labels=yb)
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
    scaler.step(opt)
    scaler.update()

    if step % eval_every == 0:
        val_loss, val_ppl = eval_loss(model, val_loader)
        print(f"step {step:04d} | train loss {loss.item():.4f} | val loss {val_loss:.4f} ppl {val_ppl:.2f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
    print("Loaded best checkpoint val loss:", best_val)

step 0200 | train loss 5.5442 | val loss 6.2511 ppl 518.57
step 0400 | train loss 3.9368 | val loss 7.0861 ppl 1195.20
step 0600 | train loss 2.4940 | val loss 7.8404 ppl 2541.15
step 0800 | train loss 1.4669 | val loss 8.3525 ppl 4240.59
step 1000 | train loss 0.8092 | val loss 8.6364 ppl 5633.20
step 1200 | train loss 0.4700 | val loss 8.8991 ppl 7325.37
step 1400 | train loss 0.2828 | val loss 9.2210 ppl 10106.80
Loaded best checkpoint val loss: 6.251069325667161


In [39]:
from collections import Counter

PAD, UNK = "<PAD>", "<UNK>"
val_frac = 0.10
n = len(tokens_story)
n_val = int(n * val_frac)
train_tokens = tokens_story[:-n_val]
val_tokens   = tokens_story[-n_val:]

# vocab from ALL tokens for coverage (train+val)
counter_all = Counter(tokens_story)
vocab = sorted(counter_all.keys(), key=lambda t: (-counter_all[t], t))

itos = [PAD, UNK] + vocab
stoi = {t:i for i,t in enumerate(itos)}

def encode(tok_list): return [stoi.get(t, stoi[UNK]) for t in tok_list]
def decode(id_list): return [itos[i] for i in id_list]

train_ids = encode(train_tokens)
val_ids   = encode(val_tokens)

print("Vocab size:", len(itos))
print("Val UNK rate:", sum(1 for t in val_tokens if t not in stoi)/len(val_tokens))

Vocab size: 6475
Val UNK rate: 0.0


In [40]:
from transformers import GPT2Config, GPT2LMHeadModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

config = GPT2Config(
    vocab_size=len(itos),
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=192,
    n_layer=3,
    n_head=3,
    resid_pdrop=0.35,
    embd_pdrop=0.35,
    attn_pdrop=0.35,
)

model = GPT2LMHeadModel(config).to(device)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

params: 2.60M


In [41]:
import math
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler

scaler = GradScaler(enabled=(device=="cuda"))
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def eval_loss(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device_type="cuda", enabled=(device=="cuda")):
                out = model(input_ids=xb, labels=yb)
                loss = out.loss
            B, T = yb.shape
            total_loss += loss.item() * (B*T)
            total_tokens += (B*T)
    avg = total_loss / total_tokens
    ppl = math.exp(avg) if avg < 20 else float("inf")
    return avg, ppl

max_steps = 400
eval_every = 50
clip_grad = 1.0

model.train()
best_val = float("inf")
best_state = None

it = iter(train_loader)
for step in range(1, max_steps+1):
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(train_loader)
        xb, yb = next(it)

    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", enabled=(device=="cuda")):
        out = model(input_ids=xb, labels=yb)
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
    scaler.step(opt)
    scaler.update()

    if step % eval_every == 0:
        val_loss, val_ppl = eval_loss(model, val_loader)
        print(f"step {step:04d} | train loss {loss.item():.4f} | val loss {val_loss:.4f} ppl {val_ppl:.2f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
    print("Loaded best checkpoint val loss:", best_val)

step 0050 | train loss 6.3900 | val loss 6.4578 ppl 637.68
step 0100 | train loss 6.2175 | val loss 6.3971 ppl 600.11
step 0150 | train loss 5.9385 | val loss 6.2600 ppl 523.23
step 0200 | train loss 5.6004 | val loss 6.2482 ppl 517.07
step 0250 | train loss 5.3047 | val loss 6.2447 ppl 515.26
step 0300 | train loss 5.0599 | val loss 6.3054 ppl 547.54
step 0350 | train loss 4.6884 | val loss 6.4380 ppl 625.16
step 0400 | train loss 4.5010 | val loss 6.5867 ppl 725.37
Loaded best checkpoint val loss: 6.244671046733856


In [42]:
import torch

def sample_next_token(logits, temperature=0.9, top_k=50, top_p=0.95):
    logits = logits / max(temperature, 1e-6)
    probs = torch.softmax(logits, dim=-1)

    if top_k and top_k > 0:
        v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
        mask = torch.zeros_like(probs)
        mask[idx] = v
        probs = mask / mask.sum()

    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cum > top_p
        cutoff[..., 0] = False
        sorted_probs[cutoff] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        return sorted_idx[torch.multinomial(sorted_probs, 1)].item()

    return torch.multinomial(probs, 1).item()

In [43]:
@torch.no_grad()
def generate(prompt, max_new_tokens=220, temperature=0.9, top_k=50, top_p=0.95):
    model.eval()
    prompt_toks = tokenize(prompt)
    ids = torch.tensor([encode(prompt_toks)], device=device, dtype=torch.long)

    for _ in range(max_new_tokens):
        x = ids[:, -block_size:]
        logits = model(input_ids=x).logits  # (1, T, V)
        next_id = sample_next_token(logits[0, -1], temperature, top_k, top_p)
        ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)

    return decode(ids[0].tolist())

seed = input("Enter a few words (prompt): ").strip() or "Poirot"
out = generate(seed, max_new_tokens=220, temperature=0.8, top_k=30, top_p=0.9)
print("\n--- Generated ---\n")
print(detokenize(out))

Enter a few words (prompt): the man jumped from the window

--- Generated ---

the man jumped from the window and a as. was. was was and The the of in of. was in.. Poirot his and. the of, You to, a an a. You in. was. was the of, The in, but not,, was -.. I... the,, the. had was in in any the. I with and was. I to. You there of, in of The the of and the as, — to a of., the was in can and The. The a, —. was in as. I. The and, the as.. “.’ a, a of. — you, a.. is, a of. I one a. “ it to, you a of. “’,, a to. “ up not to that was in in he “,.’. Poirot a. I a. I you not to the’ the. I to? “,’’ I it time it? “? “,. it” I now you it it” Poirot. I you” She the’


In [44]:
def normalize_text(s: str) -> str:
    # normalise common Gutenberg punctuation quirks
    s = s.replace("’", "'").replace("‘", "'")
    s = s.replace("“", '"').replace("”", '"')
    s = s.replace("—", "-").replace("–", "-")
    s = s.replace("\ufeff", "")  # BOM if any

    # collapse whitespace
    s = re.sub(r"\s+", " ", s)
    return s.strip()

text_clean = normalize_text(text_story).lower()

print("Preview:\n", text_clean[:600])

tokens_clean = tokenize(text_clean)
print("Num tokens (clean):", len(tokens_clean))
print("First 60:", tokens_clean[:60])

Preview:
 i the adventure of "the western star" ii the tragedy at marsdon manor iii the adventure of the cheap flat iv the mystery of hunter's lodge v the million dollar bond robbery vi the adventure of the egyptian tomb vii jewel robbery at the _grand metropolitan_ viii the kidnapped prime minister ix the disappearance of mr. davenheim x the adventure of the italian nobleman xi the case of the missing will poirot investigates poirot investigates i the adventure of "the western star" i was standing at the window of poirot's rooms looking out idly on the street below. "that's queer," i ejaculated suddenl
Num tokens (clean): 68316
First 60: ['i', 'the', 'adventure', 'of', '"', 'the', 'western', 'star', '"', 'ii', 'the', 'tragedy', 'at', 'marsdon', 'manor', 'iii', 'the', 'adventure', 'of', 'the', 'cheap', 'flat', 'iv', 'the', 'mystery', 'of', 'hunter', "'", 's', 'lodge', 'v', 'the', 'million', 'dollar', 'bond', 'robbery', 'vi', 'the', 'adventure', 'of', 'the', 'egyptian', 'tomb', 'vii', '

In [45]:
val_frac = 0.10
n = len(tokens_clean)
n_val = int(n * val_frac)

train_tokens = tokens_clean[:-n_val]
val_tokens   = tokens_clean[-n_val:]

In [46]:
from collections import Counter
PAD, UNK = "<PAD>", "<UNK>"

# vocab from ALL tokens for coverage
counter_all = Counter(tokens_clean)
vocab = sorted(counter_all.keys(), key=lambda t: (-counter_all[t], t))
itos = [PAD, UNK] + vocab
stoi = {t:i for i,t in enumerate(itos)}

def encode(tok_list): return [stoi.get(t, stoi[UNK]) for t in tok_list]
def decode(id_list): return [itos[i] for i in id_list]

train_ids = encode(train_tokens)
val_ids   = encode(val_tokens)

print("Vocab size:", len(itos))
print("Val UNK rate:", sum(1 for t in val_tokens if t not in stoi)/len(val_tokens))

block_size = 128
train_ds = LMDataset(train_ids, block_size)
val_ds   = LMDataset(val_ids, block_size)

from torch.utils.data import DataLoader
batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

print("Train windows:", len(train_ds), "Val windows:", len(val_ds))

Vocab size: 5943
Val UNK rate: 0.0
Train windows: 61356 Val windows: 6702


# Training the final model

In [47]:
# ====== 1) Load + strip Gutenberg boilerplate ======
def strip_gutenberg_boilerplate(text: str) -> str:
    start_markers = [
        "*** START OF THIS PROJECT GUTENBERG EBOOK",
        "***START OF THIS PROJECT GUTENBERG EBOOK",
        "*** START OF THE PROJECT GUTENBERG EBOOK",
    ]
    end_markers = [
        "*** END OF THIS PROJECT GUTENBERG EBOOK",
        "***END OF THIS PROJECT GUTENBERG EBOOK",
        "*** END OF THE PROJECT GUTENBERG EBOOK",
    ]

    start_idx = 0
    for m in start_markers:
        i = text.find(m)
        if i != -1:
            nl = text.find("\n", i)
            start_idx = nl if nl != -1 else i
            break

    end_idx = len(text)
    for m in end_markers:
        i = text.find(m)
        if i != -1:
            end_idx = i
            break

    return text[start_idx:end_idx].strip()

raw_text = DATA_PATH.read_text(encoding="utf-8", errors="replace")
text = strip_gutenberg_boilerplate(raw_text)

print("Chars raw   :", len(raw_text))
print("Chars cleaned:", len(text))
print("\nPreview (cleaned):\n", text[:800])


Chars raw   : 296991
Chars cleaned: 296886

Preview (cleaned):
 POIROT INVESTIGATES




  BY THE SAME AUTHOR

  THE MYSTERIOUS AFFAIR AT STYLES

  THE SECRET ADVERSARY

  THE MURDER ON THE LINKS

  THE BODLEY HEAD




  POIROT INVESTIGATES

  BY AGATHA CHRISTIE




  LONDON

  JOHN LANE THE BODLEY HEAD LIMITED




  First published in Great Britain by
  John Lane Company, The Bodley Head Limited, 1924

  Copyright © 1924 Agatha Christie Limited




  CONTENTS


  I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The Adventure of the Egyptian Tomb

  VII Jewel Robbery at the _Grand Metropolitan_

  VIII The Kidnapped Prime Minister

  IX The Disappearance of Mr. Davenheim

  X The Adventure of the Italian Nobleman



In [48]:
# ====== 2) Deterministic story trim with debug ======
def find_story_start_with_debug(text: str) -> str:
    # Start searching after CONTENTS if present
    m = re.search(r"\bCONTENTS\b", text)
    start = m.end() if m else 0
    tail = text[start:]

    # Anchor near the contents list
    mm = re.search(r"\nI\s+The\s+Adventure\s+of", tail)
    anchor = mm.start() if mm else 0
    tail2 = tail[anchor:]
    lines = tail2.splitlines()

    def narrative_like(line: str) -> bool:
        if len(line) < 30:
            return False
        lc = sum(ch.islower() for ch in line)
        return lc >= 20 and (" " in line)

    candidates = [i for i, l in enumerate(lines) if narrative_like(l)]
    candidates = candidates[:5]

    print("\nFound CONTENTS:", m is not None)
    print("Found 'I The Adventure of' anchor:", mm is not None)

    print("\nFirst 25 lines after anchor:")
    for i in range(min(25, len(lines))):
        print(f"{i:02d}: {lines[i][:120]}")

    print("\nCandidate narrative indices:", candidates)
    for idx in candidates[:3]:
        lo, hi = max(0, idx - 3), min(len(lines), idx + 6)
        print("\n--- Candidate context ---")
        for k in range(lo, hi):
            print(f"{k:04d}: {lines[k]}")

    # Default cut: a couple lines before first narrative-like line
    if candidates:
        cut = max(0, candidates[0] - 2)
        story = "\n".join(lines[cut:]).strip()
        return story

    # Fallback: return whatever we have
    return tail2.strip()

text_story = find_story_start_with_debug(text)

print("\n==== STORY PREVIEW (should look like real prose) ====\n")
print(text_story[:1200])


Found CONTENTS: True
Found 'I The Adventure of' anchor: False

First 25 lines after anchor:
00: 
01: 
02: 
03:   I The Adventure of “The Western Star”
04: 
05:   II The Tragedy at Marsdon Manor
06: 
07:   III The Adventure of the Cheap Flat
08: 
09:   IV The Mystery of Hunter’s Lodge
10: 
11:   V The Million Dollar Bond Robbery
12: 
13:   VI The Adventure of the Egyptian Tomb
14: 
15:   VII Jewel Robbery at the _Grand Metropolitan_
16: 
17:   VIII The Kidnapped Prime Minister
18: 
19:   IX The Disappearance of Mr. Davenheim
20: 
21:   X The Adventure of the Italian Nobleman
22: 
23:   XI The Case of the Missing Will
24: 

Candidate narrative indices: [3, 5, 7, 9, 11]

--- Candidate context ---
0000: 
0001: 
0002: 
0003:   I The Adventure of “The Western Star”
0004: 
0005:   II The Tragedy at Marsdon Manor
0006: 
0007:   III The Adventure of the Cheap Flat
0008: 

--- Candidate context ---
0002: 
0003:   I The Adventure of “The Western Star”
0004: 
0005:   II The Tragedy at Marsdon Man

In [49]:
# ====== 3) Normalise + lowercase ======
def normalize_text(s: str) -> str:
    s = s.replace("’", "'").replace("‘", "'")
    s = s.replace("“", '"').replace("”", '"')
    s = s.replace("—", "-").replace("–", "-")
    s = s.replace("\ufeff", "")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

text_clean = normalize_text(text_story).lower()

print("\n==== CLEAN STORY PREVIEW ====\n")
print(text_clean[:1200])


==== CLEAN STORY PREVIEW ====

i the adventure of "the western star" ii the tragedy at marsdon manor iii the adventure of the cheap flat iv the mystery of hunter's lodge v the million dollar bond robbery vi the adventure of the egyptian tomb vii jewel robbery at the _grand metropolitan_ viii the kidnapped prime minister ix the disappearance of mr. davenheim x the adventure of the italian nobleman xi the case of the missing will poirot investigates poirot investigates i the adventure of "the western star" i was standing at the window of poirot's rooms looking out idly on the street below. "that's queer," i ejaculated suddenly beneath my breath. "what is, _mon ami_?" asked poirot placidly, from the depths of his comfortable chair. "deduce, poirot, from the following facts! here is a young lady, richly dressed-fashionable hat, magnificent furs. she is coming along slowly, looking up at the houses as she goes. unknown to her, she is being shadowed by three men and a middle-aged woman. the

In [50]:
# ====== 4) Hard-trim to real narrative start (deterministic anchor) ======
# We will start from the first occurrence of this exact sentence fragment.
ANCHOR = "i was standing at the window of poirot's rooms"

idx = text_clean.find(ANCHOR)
assert idx != -1, "Anchor not found. Print a larger preview and we will adjust the anchor."
text_narr = text_clean[idx:].strip()

print("Chars clean story:", len(text_clean))
print("Chars narrative  :", len(text_narr))
print("\n==== NARRATIVE PREVIEW (should start with the sentence) ====\n")
print(text_narr[:900])

Chars clean story: 294138
Chars narrative  : 293659

==== NARRATIVE PREVIEW (should start with the sentence) ====

i was standing at the window of poirot's rooms looking out idly on the street below. "that's queer," i ejaculated suddenly beneath my breath. "what is, _mon ami_?" asked poirot placidly, from the depths of his comfortable chair. "deduce, poirot, from the following facts! here is a young lady, richly dressed-fashionable hat, magnificent furs. she is coming along slowly, looking up at the houses as she goes. unknown to her, she is being shadowed by three men and a middle-aged woman. they have just been joined by an errand boy who points after the girl, gesticulating as he does so. what drama is this being played? is the girl a crook, and are the shadowers detectives preparing to arrest her? or are _they_ the scoundrels, and are they plotting to attack an innocent victim? what does the great detective say?" "the great detective, _mon ami_, chooses, as ever, the simplest cours

In [51]:
# ====== 5) Tokenise word + punctuation ======
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(s: str):
    return TOKEN_RE.findall(s)

tokens = tokenize(text_narr)
print("\nNum tokens:", len(tokens))
print("First 80 tokens:", tokens[:80])


Num tokens: 68227
First 80 tokens: ['i', 'was', 'standing', 'at', 'the', 'window', 'of', 'poirot', "'", 's', 'rooms', 'looking', 'out', 'idly', 'on', 'the', 'street', 'below', '.', '"', 'that', "'", 's', 'queer', ',', '"', 'i', 'ejaculated', 'suddenly', 'beneath', 'my', 'breath', '.', '"', 'what', 'is', ',', '_mon', 'ami_', '?', '"', 'asked', 'poirot', 'placidly', ',', 'from', 'the', 'depths', 'of', 'his', 'comfortable', 'chair', '.', '"', 'deduce', ',', 'poirot', ',', 'from', 'the', 'following', 'facts', '!', 'here', 'is', 'a', 'young', 'lady', ',', 'richly', 'dressed', '-', 'fashionable', 'hat', ',', 'magnificent', 'furs', '.', 'she', 'is']


In [52]:
# ====== 6) Train/val split (tail holdout) ======
val_frac = 0.10
n = len(tokens)
n_val = int(n * val_frac)

train_tokens = tokens[:-n_val]
val_tokens   = tokens[-n_val:]

print("\nTrain tokens:", len(train_tokens))
print("Val tokens  :", len(val_tokens))
print("Val starts  :", val_tokens[:40])


Train tokens: 61405
Val tokens  : 6822
Val starts  : ['the', 'story', 'he', 'had', 'to', 'tell', 'was', 'a', 'sensational', 'one', '.', 'on', 'the', 'previous', 'morning', ',', 'two', 'gentlemen', 'had', 'called', 'to', 'see', 'his', 'master', '.', 'they', 'were', 'italians', ',', 'and', 'the', 'elder', 'of', 'the', 'two', ',', 'a', 'man', 'of', 'about']


In [53]:
# ====== 7) Vocab (coverage-focused; built on ALL tokens to minimise <UNK>) ======
from collections import Counter
PAD, UNK = "<PAD>", "<UNK>"

counter_all = Counter(tokens)
vocab = sorted(counter_all.keys(), key=lambda t: (-counter_all[t], t))

itos = [PAD, UNK] + vocab
stoi = {t:i for i,t in enumerate(itos)}

def encode(tok_list): return [stoi.get(t, stoi[UNK]) for t in tok_list]
def decode(id_list): return [itos[i] for i in id_list]

train_ids = encode(train_tokens)
val_ids   = encode(val_tokens)

val_unk_rate = sum(1 for t in val_tokens if t not in stoi) / len(val_tokens)
print("\nVocab size:", len(itos))
print("Val UNK rate:", val_unk_rate)


Vocab size: 5943
Val UNK rate: 0.0


In [54]:
# ====== 8) Dataset + DataLoaders ======
import torch
from torch.utils.data import Dataset, DataLoader

class LMDataset(Dataset):
    def __init__(self, ids, block_size: int):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size - 1)

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.block_size]
        y = self.ids[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 128
batch_size = 64

train_ds = LMDataset(train_ids, block_size)
val_ds   = LMDataset(val_ids, block_size)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)

print("\nTrain windows:", len(train_ds), "Val windows:", len(val_ds))


Train windows: 61276 Val windows: 6693


In [55]:
# ====== 9) HF GPT-2 style model (random init, small, regularised) ======
!pip -q install transformers

import math
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from transformers import GPT2Config, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("\ndevice:", device)

config = GPT2Config(
    vocab_size=len(itos),
    n_positions=block_size,
    n_ctx=block_size,
    n_embd=192,
    n_layer=3,
    n_head=3,
    resid_pdrop=0.35,
    embd_pdrop=0.35,
    attn_pdrop=0.35,
)

model = GPT2LMHeadModel(config).to(device)
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M")

scaler = GradScaler(enabled=(device=="cuda"))
opt = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def eval_loss(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device_type="cuda", enabled=(device=="cuda")):
                out = model(input_ids=xb, labels=yb)
                loss = out.loss
            B, T = yb.shape
            total_loss += loss.item() * (B*T)
            total_tokens += (B*T)
    avg = total_loss / total_tokens
    ppl = math.exp(avg) if avg < 20 else float("inf")
    return avg, ppl

max_steps = 400
eval_every = 50
clip_grad = 1.0

best_val = float("inf")
best_state = None

model.train()
it = iter(train_loader)

for step in range(1, max_steps+1):
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(train_loader)
        xb, yb = next(it)

    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", enabled=(device=="cuda")):
        out = model(input_ids=xb, labels=yb)
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
    scaler.step(opt)
    scaler.update()

    if step % eval_every == 0:
        val_loss, val_ppl = eval_loss(model, val_loader)
        print(f"step {step:04d} | train loss {loss.item():.4f} | val loss {val_loss:.4f} ppl {val_ppl:.2f}")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
    print("Loaded best checkpoint val loss:", best_val)


device: cuda
params: 2.50M
step 0050 | train loss 6.1667 | val loss 6.2448 ppl 515.32
step 0100 | train loss 5.9664 | val loss 6.2130 ppl 499.18
step 0150 | train loss 5.8916 | val loss 6.1679 ppl 477.18
step 0200 | train loss 5.7118 | val loss 6.1285 ppl 458.73
step 0250 | train loss 5.4333 | val loss 6.2204 ppl 502.90
step 0300 | train loss 5.3605 | val loss 6.2392 ppl 512.44
step 0350 | train loss 5.0691 | val loss 6.2165 ppl 500.92
step 0400 | train loss 4.8537 | val loss 6.4025 ppl 603.36
Loaded best checkpoint val loss: 6.1284725941144504


In [56]:
# ====== 10) Save checkpoint to Drive (so you don't lose it on reset) ======
import json, torch
from datetime import datetime

OUT_DIR = Path("/content/drive/MyDrive/Agatha_Christie/lm_checkpoints")
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_name = "word_gpt2_" + datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_dir = OUT_DIR / run_name
ckpt_dir.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), ckpt_dir / "model.pt")
with (ckpt_dir / "config.json").open("w") as f:
    json.dump(config.to_dict(), f, indent=2)
with (ckpt_dir / "stoi.json").open("w") as f:
    json.dump(stoi, f)
with (ckpt_dir / "itos.json").open("w") as f:
    json.dump(itos, f)

print("Saved to:", ckpt_dir)

Saved to: /content/drive/MyDrive/Agatha_Christie/lm_checkpoints/word_gpt2_20260219_153912


In [57]:
# ====== 11) Generation (sampling) ======
def detokenize(tok_list):
    out = []
    no_space_before = {".", ",", "!", "?", ";", ":", ")", '"', "'"}
    no_space_after  = {"(", '"'}
    for t in tok_list:
        if not out:
            out.append(t)
            continue
        if t in no_space_before:
            out[-1] = out[-1] + t
        elif out[-1] in no_space_after:
            out[-1] = out[-1] + t
        else:
            out.append(" " + t)
    return "".join(out)

import torch

def sample_next_token(logits, temperature=0.8, top_k=30, top_p=0.9):
    logits = logits / max(temperature, 1e-6)
    probs = torch.softmax(logits, dim=-1)

    if top_k and top_k > 0:
        v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
        mask = torch.zeros_like(probs)
        mask[idx] = v
        probs = mask / mask.sum()

    if top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_probs, dim=-1)
        cutoff = cum > top_p
        cutoff[..., 0] = False
        sorted_probs[cutoff] = 0.0
        sorted_probs = sorted_probs / sorted_probs.sum()
        return sorted_idx[torch.multinomial(sorted_probs, 1)].item()

    return torch.multinomial(probs, 1).item()

@torch.no_grad()
def generate(prompt, max_new_tokens=220, temperature=0.8, top_k=30, top_p=0.9):
    model.eval()
    prompt_toks = tokenize(prompt.lower())
    ids = torch.tensor([encode(prompt_toks)], device=device, dtype=torch.long)

    for _ in range(max_new_tokens):
        x = ids[:, -block_size:]
        logits = model(input_ids=x).logits
        next_id = sample_next_token(logits[0, -1], temperature, top_k, top_p)
        ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)

    return decode(ids[0].tolist())

seed = input("Enter a few words (prompt): ").strip() or "poirot"
out = generate(seed, max_new_tokens=260, temperature=0.8, top_k=30, top_p=0.9)
print("\n--- Generated ---\n")
print(detokenize(out))

Enter a few words (prompt): The man jumped from the window

--- Generated ---

the man jumped from the window in." we that is, the, he me.,'. the., he to to the, poirot you" poirot i the. he and i was it to to to, you.,'.", you not a. the of, i. i i,, i the..," the?"'. he a.,',.. i"' he in!" it. is of' the. i that to.' was, he", that,, i i.' is not you a of." you not..,' i? i to,.,, said he to - he that i' in that was that not the not?", -, the you,.?" was.', -. i,'. i you? it the not to, said -, it.?'?, i'', i.?. the the the to the the. you of you the - to,,, the - is was to,, poirot, it not''?.. - the'"' i is the. poirot in. a..,, is of,.. of,,. that.,. was.,, is" is


In [64]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

class LSTMLM(nn.Module):
    def __init__(self, vocab_size, d_emb=192, d_hid=384, n_layers=2, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_emb)
        self.lstm = nn.LSTM(
            input_size=d_emb,
            hidden_size=d_hid,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_hid, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        h, _ = self.lstm(e)
        h = self.drop(h)
        return self.head(h)

lstm_model = LSTMLM(vocab_size=len(itos)).to(device)
print("params:", f"{sum(p.numel() for p in lstm_model.parameters())/1e6:.2f}M")

params: 5.50M


In [65]:
import math
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler

scaler = GradScaler(enabled=(device=="cuda"))

opt = optim.AdamW(lstm_model.parameters(), lr=2e-4, weight_decay=0.03)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=0)

def eval_epoch(model, loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            with autocast(device_type="cuda", enabled=(device=="cuda")):
                logits = model(xb)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))
            B, T = yb.shape
            total_loss += loss.item() * (B*T)
            total_tokens += (B*T)
    avg = total_loss / total_tokens
    ppl = math.exp(avg) if avg < 20 else float("inf")
    return avg, ppl

max_epochs = 20
clip_grad = 1.0

best_val = float("inf")
best_state = None
best_epoch = None
patience = 1
bad_epochs = 0

for ep in range(1, max_epochs+1):
    lstm_model.train()
    last_train_loss = None

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=(device=="cuda")):
            logits = lstm_model(xb)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), clip_grad)
        scaler.step(opt)
        scaler.update()

        last_train_loss = loss.item()

    val_loss, val_ppl = eval_epoch(lstm_model, val_loader)
    scheduler.step(val_loss)

    lr_now = opt.param_groups[0]["lr"]
    print(f"epoch {ep:02d} | lr {lr_now:.2e} | train {last_train_loss:.4f} | val {val_loss:.4f} ppl {val_ppl:.2f}")

    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_epoch = ep
        best_state = {k: v.detach().cpu().clone() for k,v in lstm_model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs > patience:
            print(f"Early stopping at epoch {ep:02d}. Best epoch {best_epoch:02d} val {best_val:.4f}")
            break

# restore best
lstm_model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
print(f"Restored best epoch {best_epoch} with val loss {best_val:.4f} (ppl {math.exp(best_val):.2f})")

epoch 01 | lr 2.00e-04 | train 4.8989 | val 5.6166 ppl 274.96
epoch 02 | lr 1.00e-04 | train 4.2125 | val 5.6375 ppl 280.77
epoch 03 | lr 5.00e-05 | train 4.0140 | val 5.8283 ppl 339.76
Early stopping at epoch 03. Best epoch 01 val 5.6166
Restored best epoch 1 with val loss 5.6166 (ppl 274.96)


In [66]:
import json, torch

OUT_DIR = Path("/content/drive/MyDrive/Agatha_Christie/lm_checkpoints")
OUT_DIR.mkdir(parents=True, exist_ok=True)
run_name = "word_lstm_" + datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_dir = OUT_DIR / run_name
ckpt_dir.mkdir(parents=True, exist_ok=True)

torch.save(lstm_model.state_dict(), ckpt_dir / "model.pt")
with (ckpt_dir / "meta.json").open("w") as f:
    json.dump(
        {"model":"LSTMLM", "vocab_size": len(itos), "block_size": block_size},
        f, indent=2
    )
with (ckpt_dir / "stoi.json").open("w") as f:
    json.dump(stoi, f)
with (ckpt_dir / "itos.json").open("w") as f:
    json.dump(itos, f)

print("Saved to:", ckpt_dir)

Saved to: /content/drive/MyDrive/Agatha_Christie/lm_checkpoints/word_lstm_20260219_155442


In [67]:
import torch

def sample_next(logits, temperature=0.8, top_k=40):
    logits = logits / max(temperature, 1e-6)
    probs = torch.softmax(logits, dim=-1)
    if top_k and top_k > 0:
        v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
        mask = torch.zeros_like(probs)
        mask[idx] = v
        probs = mask / mask.sum()
    return torch.multinomial(probs, 1).item()

@torch.no_grad()
def generate_lstm(prompt, max_new_tokens=220, temperature=0.8, top_k=40):
    lstm_model.eval()
    toks = tokenize(prompt.lower())
    ids = [stoi.get(t, stoi[UNK]) for t in toks]
    x = torch.tensor(ids, device=device, dtype=torch.long).unsqueeze(0)

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits = lstm_model(x_cond)
        next_id = sample_next(logits[0, -1], temperature=temperature, top_k=top_k)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)

    out_toks = [itos[i] for i in x[0].tolist()]
    return out_toks

seed = input("Enter a few words (prompt): ").strip() or "poirot"
out = generate_lstm(seed, max_new_tokens=260, temperature=0.8, top_k=40)
print("\n--- Generated ---\n")
print(detokenize(out))

Enter a few words (prompt): the man jumped from the window

--- Generated ---

the man jumped from the window," i had been your way in the few in some man - lady."" i were very in me.""" then, the little is of the own way. the doctor is?" i," she not, but if and said, there was at the friend, and i can - a first way to be a doctor, and did no!" it were not a small." i must will that,"" said you," i got his own minister." ah?""" he' d say that i will be?"" i was see," the great friend of a same' s a great way, and they have just i have his way, but the one had' t been a diamond." what was have that at the moment of he' s death.""" he is it, but the first, my had had a doctor of the friend of his - - we were you were one in and night. i am?"" i were not." the few, not. i cried a man of the flat." i think?""" he is to have our good on as a case to been once." said you did not go and that poirot' s good in the - of i


In [68]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def eval_accuracy(model, loader, topk=(1,5)):
    model.eval()
    correct = {k: 0 for k in topk}
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)  # (B,T,V)
        B, T, V = logits.shape
        logits = logits.reshape(-1, V)
        y = yb.reshape(-1)

        total += y.numel()

        for k in topk:
            pred = logits.topk(k, dim=-1).indices  # (N,k)
            correct[k] += (pred == y.unsqueeze(-1)).any(dim=-1).sum().item()

    return {k: correct[k] / total for k in topk}

acc = eval_accuracy(lstm_model, val_loader, topk=(1,5))
acc

{1: 0.16409653883713943, 5: 0.35057889498197115}

In [69]:
import torch
import torch.nn.functional as F
from collections import Counter

UNK_ID = stoi.get("<UNK>", None)

def apply_repetition_penalty(logits, generated_ids, penalty=1.15, window=128):
    # Penalize tokens that appeared recently to reduce loops
    if penalty <= 1.0:
        return logits
    recent = generated_ids[-window:]
    counts = Counter(recent)
    for tok_id, c in counts.items():
        # divide positive logits / multiply negative logits slightly
        if logits[tok_id] > 0:
            logits[tok_id] = logits[tok_id] / (penalty ** c)
        else:
            logits[tok_id] = logits[tok_id] * (penalty ** c)
    return logits

def nucleus_sample(probs, top_p=0.9):
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    cutoff = cum > top_p
    cutoff[..., 0] = False
    sorted_probs[cutoff] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum()
    next_id = sorted_idx[torch.multinomial(sorted_probs, 1)].item()
    return next_id

@torch.no_grad()
def generate_lstm_better(prompt, max_new_tokens=260, temperature=0.75, top_p=0.9, top_k=0,
                         repetition_penalty=1.15, ban_unk=True):
    lstm_model.eval()
    prompt_toks = tokenize(prompt.lower())
    ids = [stoi.get(t, stoi["<UNK>"]) for t in prompt_toks]
    x = torch.tensor([ids], device=device, dtype=torch.long)

    generated = ids[:]  # list of ids

    for _ in range(max_new_tokens):
        x_cond = x[:, -block_size:]
        logits = lstm_model(x_cond)[0, -1].float()  # (V,)

        # repetition penalty
        logits = apply_repetition_penalty(logits, generated, penalty=repetition_penalty, window=128)

        # ban UNK
        if ban_unk and UNK_ID is not None:
            logits[UNK_ID] = -1e9

        # temperature
        logits = logits / max(temperature, 1e-6)
        probs = F.softmax(logits, dim=-1)

        # optional top-k
        if top_k and top_k > 0:
            v, idx = torch.topk(probs, k=min(top_k, probs.size(-1)))
            mask = torch.zeros_like(probs)
            mask[idx] = v
            probs = mask / mask.sum()

        # top-p sampling
        next_id = nucleus_sample(probs, top_p=top_p)

        generated.append(next_id)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)

    out_tokens = [itos[i] for i in generated]
    return out_tokens

seed = input("Enter a few words (prompt): ").strip() or "poirot"
out = generate_lstm_better(seed, max_new_tokens=260, temperature=0.75, top_p=0.9,
                           repetition_penalty=1.15, ban_unk=True)
print("\n--- Generated ---\n")
print(detokenize(out))

Enter a few words (prompt): the man jumped from the window

--- Generated ---

the man jumped from the window and all, that, i have been slight dignified of the police minister."" poirot? you will be not be his search - her, is a end of london." we think my small gifted to a parcel in a traitor; he then in some station. well! she' t know that a discovery - scotland did were this eyes with his man to it which in too here! i was still thing has do that was flats with a premium on an card of force by the boathouse on your watch. but as don poirot, but what said no depart at the door? and cried here - its would your dressing of to she' s girl as • it are been up and perhaps so money out and you had no shook the good investigation, hastings from all that why to." ah," a bedroom were along her his case? how poirot is sir. for my is' ve were notice in him at me us from once at for some behind of the york than it - consulting by our door for my local man, hastings: he handed to a fact' s young